In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

In [ ]:
!pip install transformers -q

import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

print("✅ All imports done")

In [ ]:
BASE = '/content/drive/MyDrive/MultimodalSentiment/data/'

train_df = pd.read_csv(BASE + 'train_sent_emo.csv')
val_df   = pd.read_csv(BASE + 'dev_sent_emo.csv')
test_df  = pd.read_csv(BASE + 'test_sent_emo.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}

class MELDDataset(Dataset):
    def __init__(self, df, max_len=128):
        self.df = df[df['Sentiment'].isin(label_map.keys())].reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['Utterance'])
        enc = tokenizer(text, max_length=self.max_len,
                        padding='max_length', truncation=True,
                        return_tensors='pt')
        input_ids      = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()
        image          = torch.zeros(3, 224, 224)
        label          = torch.tensor(label_map[row['Sentiment']], dtype=torch.long)
        return input_ids, attention_mask, image, label

train_ds = MELDDataset(train_df)
val_ds   = MELDDataset(val_df)
test_ds  = MELDDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

print(f"✅ DataLoaders ready")

In [ ]:
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # ── Text encoder (BERT) ──
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        # Freeze all BERT layers except last 2
        for name, param in self.bert.named_parameters():
            if 'encoder.layer.11' not in name and 'pooler' not in name:
                param.requires_grad = False

        # ── Image encoder (ResNet-50) ──
        resnet = models.resnet50(pretrained=True)
        for name, param in resnet.named_parameters():
            if 'layer4' not in name:
                param.requires_grad = False
        # Remove final FC — keep feature extractor only
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])
        # Output shape: (batch, 2048, 1, 1)

        # ── Fusion classifier ──
        # 768 (BERT) + 2048 (ResNet) = 2816
        self.fusion = nn.Sequential(
            nn.Linear(2816, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        # Text features
        bert_out  = self.bert(input_ids=input_ids,
                              attention_mask=attention_mask)
        text_feat = bert_out.pooler_output          # (batch, 768)

        # Image features
        img_feat  = self.image_encoder(images)
        img_feat  = img_feat.view(img_feat.size(0), -1)  # (batch, 2048)

        # Concatenate both
        combined  = torch.cat([text_feat, img_feat], dim=1)  # (batch, 2816)

        return self.fusion(combined)

fusion_model = MultimodalFusionModel()
trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f"✅ Fusion model built")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fusion_model = fusion_model.to(device)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, fusion_model.parameters()), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()

train_losses, val_accs = [], []
best_val_acc   = 0
patience_count = 0

print(f"Training Fusion Model on: {device}")
print("="*55)

for epoch in range(10):
    # ── Train ──
    fusion_model.train()
    total_loss = 0

    for input_ids, attention_mask, images, labels in train_loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images         = images.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        outputs = fusion_model(input_ids, attention_mask, images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ── Validate ──
    fusion_model.eval()
    preds, true = [], []

    with torch.no_grad():
        for input_ids, attention_mask, images, labels in val_loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            images         = images.to(device)
            labels         = labels.to(device)
            outputs        = fusion_model(input_ids, attention_mask, images)
            preds         += outputs.argmax(1).cpu().tolist()
            true          += labels.cpu().tolist()

    val_acc = accuracy_score(true, preds)
    val_accs.append(val_acc)
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1:2d}/10 | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        patience_count = 0
        torch.save(fusion_model.state_dict(),
                   '/content/drive/MyDrive/MultimodalSentiment/models/best_fusion.pth')
        print(f"  ✅ Best model saved! Acc: {best_val_acc:.4f}")
    else:
        patience_count += 1
        if patience_count >= 3:
            print(f"\n⏹ Early stopping at epoch {epoch+1}")
            break

    print("-"*55)

print(f"\n🎯 Best Fusion Val Accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, color='#378ADD', marker='o', label='Train Loss')
axes[0].set_title('Fusion Model — Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(val_accs, color='#1D9E75', marker='o', label='Val Accuracy')
axes[1].axhline(y=0.6844, color='#D85A30', linestyle='--', label='BERT baseline')
axes[1].axhline(y=0.4238, color='#BA7517', linestyle='--', label='ResNet baseline')
axes[1].set_title('Fusion Model — Val Accuracy vs Baselines')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/MultimodalSentiment/outputs/fusion_curves.png', dpi=150)
plt.show()
print("✅ Curves saved!")

In [ ]:
# Load best fusion model
fusion_model.load_state_dict(
    torch.load('/content/drive/MyDrive/MultimodalSentiment/models/best_fusion.pth'))
fusion_model.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, images, labels in test_loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images         = images.to(device)
        outputs        = fusion_model(input_ids, attention_mask, images)
        all_preds     += outputs.argmax(1).cpu().tolist()
        all_true      += labels.tolist()

# Classification report
print("Classification Report:")
print(classification_report(all_true, all_preds,
      target_names=['Negative', 'Neutral', 'Positive']))

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix — Fusion Model (Test Set)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/MultimodalSentiment/outputs/confusion_matrix.png', dpi=150)
plt.show()
print("✅ Confusion matrix saved!")